# 05: Training, Metrics, and Run Wrapper

Builds the execution layer on top of manipulation_functions.py:

1. train_model: three classifiers (LR, RF, XGBoost), fixed hyperparameters
2. compute_metrics: the seven dependent variables, computed on test only
3. run_experiment: full run wrapper with W&B config and metric logging

Closes with the 24-run pilot exercising the corner conditions before
scaling to 960 runs. Functions exported to experiment_functions.py.

In [1]:
import pandas as pd
import numpy as np
import wandb

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, recall_score, f1_score,
                             balanced_accuracy_score, matthews_corrcoef)

from manipulation_functions import *   

BUCKET = "osilesi-dissertation-data-2026"
adult = pd.read_csv(f"s3://{BUCKET}/processed/adult_final_pruned.csv")

In [6]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def train_model(train_df, classifier_name, seed):
    X = train_df.drop(columns=["target"])
    y = train_df["target"]

    if classifier_name == "logistic_regression":
        model = make_pipeline(
            StandardScaler(),
            LogisticRegression(max_iter=2000, random_state=seed))
    elif classifier_name == "random_forest":
        model = RandomForestClassifier(n_estimators=100,
                                       random_state=seed, n_jobs=-1)
    elif classifier_name == "xgboost":
        model = XGBClassifier(n_estimators=100, max_depth=6,
                              learning_rate=0.3, random_state=seed,
                              eval_metric="logloss", n_jobs=-1)
    else:
        raise ValueError(classifier_name)

    model.fit(X, y)
    return model

In [7]:
def compute_metrics(model, test_df):
    """Compute all seven dependent variables on the test set.

    Fairness metrics follow the female-minus-male convention:
    negative equal opportunity difference means the female
    subgroup has the lower true positive rate.
    """
    X = test_df.drop(columns=["target"])
    y = test_df["target"].values
    pred = model.predict(X)
    female = test_df["sex_female"].values == 1

    # Performance metrics
    m = {
        "accuracy":          accuracy_score(y, pred),
        "minority_recall":   recall_score(y, pred, pos_label=1),
        "macro_f1":          f1_score(y, pred, average="macro"),
        "balanced_accuracy": balanced_accuracy_score(y, pred),
        "mcc":               matthews_corrcoef(y, pred),
    }

    # Subgroup true positive and false negative rates
    def tpr(mask):
        pos = (y == 1) & mask
        return np.nan if pos.sum() == 0 else pred[pos].mean()

    tpr_f, tpr_m = tpr(female), tpr(~female)
    m["equal_opportunity_diff"] = tpr_f - tpr_m
    m["fnr_diff"] = (1 - tpr_f) - (1 - tpr_m)

    # Diagnostics, logged but not dependent variables
    m["n_test"] = len(y)
    m["n_female_pos_test"] = int(((y == 1) & female).sum())
    return m

In [4]:
def run_experiment(df, dataset_name, classifier_name, imbalance,
                   min_level, min_type, seed, base_n=15000,
                   project="dissertation-rq-experiments"):
    config = dict(dataset=dataset_name, classifier=classifier_name,
                  imbalance=imbalance, min_level=min_level,
                  min_type=min_type, seed=seed)

    run = wandb.init(project=project, config=config,
                     name=f"{dataset_name}-{classifier_name}-{imbalance}"
                          f"-{min_type}{min_level}-s{seed}",
                     reinit=True)
    try:
        d = apply_imbalance(df, imbalance, seed, base_n=base_n)
        if min_type == "horizontal":
            d = apply_horizontal_minimization(d, min_level, seed)
        else:
            d = apply_vertical_minimization(d, min_level, seed)
        train, test = make_split(d, seed)

        model = train_model(train, classifier_name, seed)
        metrics = compute_metrics(model, test)

        wandb.log(metrics)
        result = {**config, **metrics, "status": "ok"}
    except Exception as e:
        result = {**config, "status": f"error: {e}"}
    finally:
        run.finish()
    return result

In [8]:
pilot_conditions = [
    (imb, lvl, mtype)
    for imb in ["50/50", "95/5"]          # imbalance extremes
    for lvl in ["100", "25"]              # minimization extremes
    for mtype in ["horizontal", "vertical"]
]

results = []
for clf in ["logistic_regression", "random_forest", "xgboost"]:
    for imb, lvl, mtype in pilot_conditions:
        r = run_experiment(adult, "adult", clf, imb, lvl, mtype, seed=42)
        results.append(r)
        print(r["classifier"], imb, lvl, mtype, r["status"])

pilot = pd.DataFrame(results)
pilot.to_csv("pilot_results.csv", index=False)
pilot

accuracy,▁
balanced_accuracy,▁
equal_opportunity_diff,▁
fnr_diff,▁
macro_f1,▁
mcc,▁
minority_recall,▁
n_female_pos_test,▁
n_test,▁
accuracy,0.828
balanced_accuracy,0.828


logistic_regression 50/50 100 horizontal ok


accuracy,▁
balanced_accuracy,▁
equal_opportunity_diff,▁
fnr_diff,▁
macro_f1,▁
mcc,▁
minority_recall,▁
n_female_pos_test,▁
n_test,▁
accuracy,0.828
balanced_accuracy,0.828


logistic_regression 50/50 100 vertical ok


accuracy,▁
balanced_accuracy,▁
equal_opportunity_diff,▁
fnr_diff,▁
macro_f1,▁
mcc,▁
minority_recall,▁
n_female_pos_test,▁
n_test,▁
accuracy,0.80267
balanced_accuracy,0.80265


...

accuracy,▁
balanced_accuracy,▁
equal_opportunity_diff,▁
fnr_diff,▁
macro_f1,▁
mcc,▁
minority_recall,▁
n_female_pos_test,▁
n_test,▁
accuracy,0.94667
balanced_accuracy,0.60812


xgboost 95/5 25 horizontal ok


accuracy,▁
balanced_accuracy,▁
equal_opportunity_diff,▁
fnr_diff,▁
macro_f1,▁
mcc,▁
minority_recall,▁
n_female_pos_test,▁
n_test,▁
accuracy,0.95778
balanced_accuracy,0.63041


xgboost 95/5 25 vertical ok


,dataset,classifier,imbalance,min_level,min_type,seed,accuracy,minority_recall,macro_f1,balanced_accuracy,mcc,equal_opportunity_diff,fnr_diff,n_test,n_female_pos_test,status
0,adult,logistic_regression,50/50,100,horizontal,42,0.828000,0.839111,0.827979,0.828000,0.656162,-0.175250,0.175250,4500,339,ok
1,adult,logistic_regression,50/50,100,vertical,42,0.828000,0.839111,0.827979,0.828000,0.656162,-0.175250,0.175250,4500,339,ok
2,adult,logistic_regression,50/50,25,horizontal,42,0.802667,0.820604,0.802598,0.802651,0.605704,-0.148979,0.148979,1125,85,ok
3,adult,logistic_regression,50/50,25,vertical,42,0.820444,0.835111,0.820406,0.820444,0.641165,-0.177487,0.177487,4500,339,ok
4,adult,logistic_regression,95/5,100,horizontal,42,0.954889,0.204444,0.644272,0.599415,0.350180,-0.129048,0.129048,4500,32,ok
5,adult,logistic_regression,95/5,100,vertical,42,0.954889,0.204444,0.644272,0.599415,0.350180,-0.129048,0.129048,4500,32,ok
6,adult,logistic_regression,95/5,25,horizontal,42,0.951111,0.125000,0.588840,0.559694,0.242950,0.000000,0.000000,1125,8,ok
7,adult,logistic_regression,95/5,25,vertical,42,0.956667,0.191111,0.641842,0.594035,0.369742,-0.113504,0.113504,4500,32,ok
8,adult,random_forest,50/50,100,horizontal,42,0.816667,0.811111,0.816661,0.816667,0.633372,-0.069347,0.069347,4500,339,ok
9,adult,random_forest,50/50,100,vertical,42,0.816667,0.811111,0.816661,0.816667,0.633372,-0.069347,0.069347,4500,339,ok
